In [1]:
!pip install demucs


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torchaudio
from demucs import pretrained
from demucs.apply import apply_model
import os

# Load pretrained Demucs model
model = pretrained.get_model('htdemucs')

# Define song pairs to process
song_pairs = [
    # (folder, original_song, ai_song)
    ("song1", "kanye.mp3", "kanye_new.mp3"),
    ("song2", "wap.mp3", "wap_new.mp3"), 
    ("song3", "islandgirl.mp3", "islandgirl_new.mp3"),
    ("song4", "whiteboy.mp3", "whiteboy_new.mp3")
]

def separate_vocals(input_path, output_vocals_path, output_instruments_path):
    """Separate vocals from a song and save both vocals and instruments"""
    try:
        wav, sr = torchaudio.load(input_path)
        wav = wav.unsqueeze(0)
        sources = apply_model(model, wav, split=True, progress=True)[0]
        sources_names = model.sources
        vocals = sources[sources_names.index("vocals")]
        instruments = sum(sources[i] for i, name in enumerate(sources_names) if name != "vocals")
        torchaudio.save(output_vocals_path, vocals, sr)
        torchaudio.save(output_instruments_path, instruments, sr)
        
        print(f"✓ Successfully processed: {input_path}")
        return True
        
    except Exception as e:
        print(f"✗ Error processing {input_path}: {e}")
        return False



In [5]:
separate_vocals("../song4/whiteboy_new.mp3", "../song4/vocals_ai.wav", "../song4/instruments_ai.wav")

100%|██████████████████████████████████████████████| 280.79999999999995/280.79999999999995 [02:07<00:00,  2.21seconds/s]
c:\Users\jiyon\Desktop\Programming_projects\UNSW\musictransformationanalysis\.venv\Lib\site-packages\torchaudio\_backend\utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(


✓ Successfully processed: ../song4/whiteboy_new.mp3


True

In [ ]:
for folder, original_song, ai_song in song_pairs:
    print(f"\n--- Processing {folder} ---")
    
    folder_path = f"./{folder}"
    
    original_input = f".{folder_path}/{original_song}"
    original_vocals_output = f".{folder_path}/vocals.wav"
    original_instruments_output = f".{folder_path}/instruments.wav"
    
    if os.path.exists(original_input):
        separate_vocals(original_input, original_vocals_output, original_instruments_output)
    else:
        print(f"✗ File not found: {original_input}")
    
    ai_input = f".{folder_path}/{ai_song}"
    ai_vocals_output = f".{folder_path}/vocals_ai.wav"
    ai_instruments_output = f".{folder_path}/instruments_ai.wav"
    
    if os.path.exists(ai_input):
        separate_vocals(ai_input, ai_vocals_output, ai_instruments_output)
    else:
        print(f"✗ File not found: {ai_input}")

    